In [0]:
from pyspark.sql.functions import col

df_hist = spark.table(
    "chilecompra.bronze.ordenes_compra_historical"
)

In [0]:
from pyspark.sql.functions import expr

def decimal_col(column_name, precision=20, scale=6):
    return expr(
        f"""
        try_cast(
            replace(trim(`{column_name}`), ',', '.')
            AS DECIMAL({precision},{scale})
        )
        """
    )

In [0]:
df_items = (
    df_hist
    .select(
        col("iditem"),
        col("codigo"),

        col("codigocategoria").alias("codigo_categoria"),
        col("categoria"),

        col("codigoproductoonu").alias("codigo_producto_onu"),
        col("nombreroductogenerico").alias("nombre_producto_generico"),

        col("rubron1").alias("rubro_n1"),
        col("rubron2").alias("rubro_n2"),
        col("rubron3").alias("rubro_n3"),

        col("especificacioncomprador").alias("especificacion_comprador"),
        col("especificacionproveedor").alias("especificacion_proveedor"),

        decimal_col("cantidad").alias("cantidad"),
        col("unidadmedida").alias("unidad_medida"),

        col("monedaitem").alias("moneda"),

        decimal_col("precioneto").alias("precio_neto"),
        decimal_col("totalcargos").alias("total_cargos"),
        decimal_col("totaldescuentos").alias("total_descuentos"),
        decimal_col("totalimpuestos").alias("total_impuestos"),
        decimal_col("totallineaneto").alias("total_linea_neto"),

        col("_source_year"),
        col("_source_month")
    )
)

In [0]:
total_rows = df_items.count()

if total_rows == 0:
    raise ValueError(
        "DQ FAILED: historical items source contains 0 rows"
    )
null_iditem = (
    df_items
    .filter(col("iditem").isNull())
    .count()
)

duplicate_iditem = (
    total_rows
    - df_items.select("iditem").distinct().count()
)

null_codigo = (
    df_items
    .filter(col("codigo").isNull())
    .count()
)

if null_iditem > 0:
    raise ValueError(
        f"DQ FAILED: {null_iditem} rows have iditem NULL"
    )

if duplicate_iditem > 0:
    raise ValueError(
        f"DQ FAILED: {duplicate_iditem} duplicated iditem values"
    )

if null_codigo > 0:
    raise ValueError(
        f"DQ FAILED: {null_codigo} rows have codigo NULL"
    )

print(f"Basic item DQ passed: {total_rows} rows")

In [0]:
df_oc = spark.table(
    "chilecompra.silver.ordenes_compra"
).select("codigo")

orphan_items = (
    df_items.alias("i")
    .join(
        df_oc.alias("o"),
        on="codigo",
        how="left_anti"
    )
    .count()
)

if orphan_items > 0:
    raise ValueError(
        f"DQ FAILED: {orphan_items} items do not have a matching order"
    )

print("Referential integrity DQ passed")

In [0]:
from delta.tables import DeltaTable

target_table = "chilecompra.silver.ordenes_compra_items"

if not spark.catalog.tableExists(target_table):

    (
        df_items.write
        .format("delta")
        .saveAsTable(target_table)
    )

else:

    delta_target = DeltaTable.forName(
        spark,
        target_table
    )

    (
        delta_target.alias("t")
        .merge(
            df_items.alias("s"),
            "t.iditem = s.iditem"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
df_silver_items = spark.table(target_table)

silver_rows = df_silver_items.count()

silver_distinct = (
    df_silver_items
    .select("iditem")
    .distinct()
    .count()
)

silver_null_iditem = (
    df_silver_items
    .filter(col("iditem").isNull())
    .count()
)

missing_items = (
    df_items
    .select("iditem")
    .join(
        df_silver_items.select("iditem"),
        on="iditem",
        how="left_anti"
    )
    .count()
)

if missing_items > 0:
    raise ValueError(
        f"DQ FAILED: {missing_items} current source items "
        f"are missing from Silver"
    )

if silver_rows != silver_distinct:
    raise ValueError(
        f"DQ FAILED: rows={silver_rows}, "
        f"distinct iditem={silver_distinct}"
    )

if silver_null_iditem > 0:
    raise ValueError(
        f"DQ FAILED: {silver_null_iditem} rows have iditem NULL"
    )

print(
    f"Silver items DQ passed: "
    f"{silver_rows} items, "
    f"all current source items present"
)